# QDIST v4.1.0 — blocking-remediation analytical preflight

This notebook evaluates a **candidate** correction to the native-decoded hard-plateau detector after the v4.0 audit identified asymmetric negative-rail failure and frame-grid phase dependence.

Scientific scope is intentionally narrow: visible hard-plateau morphology in the decoded waveform. It does not establish total nonlinear distortion, soft clipping, codec distortion, automatic gain control, dynamic-range compression, overload stage, or physiological signal quality.

Decision contract:

- proposed primary feature: `qdist_hard_clipped_sample_fraction`;
- proposed secondary feature: `qdist_hard_clip_event_rate_per_min`;
- conditional/audit-only feature: `qdist_hard_clipped_frame_fraction`;
- no cohort overwrite, finalization, publication export, or freeze is permitted here;
- full-cohort recomputation, held-out real-speech injection, output inspection, and independent blinded human/technical review remain blocking.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd

PROJECT_ROOT_OVERRIDE = None
RUN_PACKAGE_TESTS = True
RUN_REMEDIATION_PREFLIGHT = True

def find_project_root():
    if PROJECT_ROOT_OVERRIDE is not None:
        return Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "src" / "paper1_qc").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate the pipeline project root.")

ROOT = find_project_root()
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "src")]

from paper1_qc_reviewed.qdist_v410_remediation import run_remediation_preflight

OUTPUT_ROOT = (
    ROOT
    / "MAIN outputs/02_FEATURE_REVIEWED/00_working_candidates"
    / "nonlinear_distortion"
    / "qdist_v410_remediation_preflight"
)
TEST_FILE = ROOT / "tests" / "test_qdist_v410_candidate.py"
print("Project root:", ROOT)
print("Candidate output root:", OUTPUT_ROOT)

Project root: C:\Users\musikicn\Desktop\Nevena_project\Paper_1\paper_1
Candidate output root: C:\Users\musikicn\Desktop\Nevena_project\Paper_1\paper_1\MAIN outputs/02_FEATURE_REVIEWED/00_working_candidates\nonlinear_distortion\qdist_v410_remediation_preflight


In [2]:
package_tests_passed = False
package_test_output = "not requested"
if RUN_PACKAGE_TESTS:
    completed = subprocess.run(
        [sys.executable, "-m", "pytest", str(TEST_FILE), "-q"],
        cwd=ROOT,
        capture_output=True,
        text=True,
        check=False,
    )
    package_test_output = (completed.stdout + "\n" + completed.stderr).strip()
    package_tests_passed = completed.returncode == 0
    print(package_test_output)
    print("Package tests passed:", package_tests_passed)

............................                                             [100%]
Package tests passed: True


In [3]:
if not RUN_REMEDIATION_PREFLIGHT:
    raise RuntimeError("RUN_REMEDIATION_PREFLIGHT must be True for this notebook.")

evidence = run_remediation_preflight(OUTPUT_ROOT)
checks = evidence["checks"]
figure_index = evidence["figure_index"]
manifest = evidence["manifest"]

display(checks[["gate", "check", "passed", "required"]])
display(figure_index)

,gate,check,passed,required
0,G1,candidate measurement identity and corrected r...,True,qdist-v4.1.0-candidate
1,G2,all feature values reconstruct from candidate ...,True,absolute error <=1e-15
2,G3,one-sided polarity inversion is exact,True,all three outputs exact
3,G3,direct burden and event rate survive arbitrary...,True,event rate and sample fraction exact
4,G3,frame-grid phase dependence is detected and pr...,True,sensitivity visible; feature conditional
5,G4,all moderate known-truth doses are detected,True,all rows with true burden >=0.001 positive
6,G4,detected moderate doses have sample precision ...,True,>=0.99
7,G4,repeated low-level saturation path is exercised,True,at least one accepted repeated_low_level_satur...
8,G5,"periodic, clean, impulse/noise, smooth saturat...",True,no synthetic-control positives


,panel,stem,png,svg,pdf,source_csv,caption,provenance
0,A,qdist_v410_panel-A_construct-response,C:\Users\musikicn\Desktop\Nevena_project\Paper...,C:\Users\musikicn\Desktop\Nevena_project\Paper...,C:\Users\musikicn\Desktop\Nevena_project\Paper...,C:\Users\musikicn\Desktop\Nevena_project\Paper...,C:\Users\musikicn\Desktop\Nevena_project\Paper...,C:\Users\musikicn\Desktop\Nevena_project\Paper...
1,B,qdist_v410_panel-B_discriminant-specificity,C:\Users\musikicn\Desktop\Nevena_project\Paper...,C:\Users\musikicn\Desktop\Nevena_project\Paper...,C:\Users\musikicn\Desktop\Nevena_project\Paper...,C:\Users\musikicn\Desktop\Nevena_project\Paper...,C:\Users\musikicn\Desktop\Nevena_project\Paper...,C:\Users\musikicn\Desktop\Nevena_project\Paper...
2,C,qdist_v410_panel-C_transformation-contract,C:\Users\musikicn\Desktop\Nevena_project\Paper...,C:\Users\musikicn\Desktop\Nevena_project\Paper...,C:\Users\musikicn\Desktop\Nevena_project\Paper...,C:\Users\musikicn\Desktop\Nevena_project\Paper...,C:\Users\musikicn\Desktop\Nevena_project\Paper...,C:\Users\musikicn\Desktop\Nevena_project\Paper...


In [4]:
failed = checks.loc[~checks["passed"].astype(bool)]
print("Blocking synthetic checks:", f"{int(checks['passed'].sum())}/{len(checks)}")
print("Generated preflight panels:", sorted(figure_index["panel"].tolist()))
if len(failed):
    display(failed)

assert len(failed) == 0, "Candidate remediation preflight has blocking synthetic failures."
assert manifest["candidate_only"] is True
assert manifest["freeze_allowed"] is False
assert manifest["feature_values_from_cohort_recomputed"] is False
assert manifest["cohort_rerun_required"] is True
assert manifest["real_speech_injection_required"] is True
assert manifest["independent_blinded_human_review_required"] is True

Blocking synthetic checks: 9/9
Generated preflight panels: ['A', 'B', 'C']


In [6]:
checklist_path = (
    ROOT
    / "notebooks/02_feature_extraction"
    / "05_QDIST"
    / "support/QDIST_Master_Validation_Checklist_v1_1_REMEDIATION.csv"
)

checklist = pd.read_csv(
    checklist_path,
    keep_default_na=False,
)

display(
    checklist.groupby("status", dropna=False)
    .size()
    .rename("item_count")
    .reset_index()
    .sort_values("status")
)

review_statuses = [
    "FAIL",
    "PENDING",
    "CONDITIONAL",
]

review_columns = [
    "gate",
    "item_id",
    "domain",
    "status",
    "requirement",
    "required_evidence",
    "evidence_path_notes",
    "reviewer_note",
]

display(
    checklist.loc[
        checklist["status"].isin(review_statuses),
        review_columns,
    ].sort_values(
        ["gate", "status", "item_id"]
    ).reset_index(drop=True)
)

,status,item_count
0,CONDITIONAL,18
1,FAIL,11
2,N/A,1
3,PASS,22
4,PENDING,8


,gate,item_id,domain,status,requirement,required_evidence,evidence_path_notes,reviewer_note
0,G1,C1,Construct,CONDITIONAL,Family construct is stated as an observable ac...,Contract/registry,src/paper1_qc/qdist.py; src/paper1_qc...,Narrow hard-plateau observable is stated in co...
1,G1,C3,Construct,CONDITIONAL,Included and excluded phenomena are explicit.,Scope table,src/paper1_qc/qdist.py; src/paper1_qc...,Detector exclusions are explicit; manuscript T...
2,G10,INT1,Interpretability,CONDITIONAL,"Name, unit, direction, support, and nonordinal...",Feature passport,src/paper1_qc/qdist.py; src/paper1_qc...,Sample and event units are clear; frame name/u...
3,G10,INT3,Interpretability,CONDITIONAL,Known confounds and failure modes are explicit.,Feature passport,src/paper1_qc/qdist.py; src/paper1_qc...,"Many confounds are listed, but asymmetric clip..."
4,G10,ML3,ML readiness,CONDITIONAL,Feature can be consumed for biomarker-specific...,ML handoff table,src/paper1_qc/qdist.py; src/paper1_qc...,"Provenance survives handoff, but default simul..."
5,G10,C6,Construct,FAIL,Family label matches what is actually measured.,Registry/final wording,work/qdist_deep_audit/QDIST_Scientific_Audit_a...,The label nonlinear distortion is broader than...
6,G10,G10,Freeze,FAIL,Every feature has retain/revise/conditional/ex...,Decision table,work/qdist_deep_audit/QDIST_Scientific_Audit_a...,Current retain/default-model decisions conflic...
7,G10,G11,Freeze,FAIL,"Executed notebook, tests, registry, figures, t...",Freeze manifest,work/qdist_deep_audit/QDIST_Scientific_Audit_a...,Freeze must remain blocked; qdist-v4.1.0 requi...
8,G10,G12,Freeze,FAIL,Manuscript wording and feature census match th...,Cross-document audit,work/qdist_deep_audit/QDIST_Scientific_Audit_a...,Manuscript reports five QGAIN/26 total indicat...
9,G10,F2,Figures,PENDING,"Each panel has SVG/PDF, high-resolution PNG, s...",Artifact inventory,work/qdist_deep_audit/QDIST_Scientific_Audit_a...,The actual 23 bundles and 60 review packages w...


In [7]:
manifest_path = (
    OUTPUT_ROOT
    / "manifests"
    / "qdist_v410_remediation_preflight_manifest.json"
)
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
manifest["package_tests_passed"] = bool(package_tests_passed)
manifest["package_test_file"] = str(TEST_FILE)
manifest["notebook_source"] = str(Path.cwd())
manifest_path.write_text(
    json.dumps(manifest, indent=2, sort_keys=True),
    encoding="utf-8",
)
print(json.dumps({
    "measurement_version": manifest["measurement_version"],
    "candidate_only": manifest["candidate_only"],
    "freeze_allowed": manifest["freeze_allowed"],
    "package_tests_passed": manifest["package_tests_passed"],
    "remaining_blockers": [
        "full-cohort rerun",
        "held-out real-speech injection",
        "external output inspection",
        "independent blinded human/technical review",
    ],
}, indent=2))

{
  "measurement_version": "qdist-v4.1.0-candidate",
  "candidate_only": true,
  "freeze_allowed": false,
  "package_tests_passed": true,
  "remaining_blockers": [
    "full-cohort rerun",
    "held-out real-speech injection",
    "external output inspection",
    "independent blinded human/technical review"
  ]
}


## Interpretation guardrail

Passing this preflight means only that the candidate correction behaves as specified on the included synthetic and algebraic checks. It is **not** evidence of clinical validity, real-world sensitivity, cohort prevalence, device-stage attribution, or readiness to freeze. The next valid step is a candidate full-cohort rerun followed by independently adjudicated positive, rejected-candidate, valid-zero, and real-speech-injection strata.